# Paper 2 · Langkah 11: arsitektur pembanding (CLIP yang dibekukan)

**Pertanyaan:** apakah tiga temuan utama juga berlaku untuk jenis detektor yang berbeda, yaitu model fondasi yang dibekukan dengan pengklasifikasi linear di atasnya?

**Model:** CLIP ViT-L/14 (bobot OpenAI), dibekukan. Fitur gambar diambil sekali, lalu regresi logistik dilatih di atasnya. Data, pembagian fold, dan tiga kondisi kompresi sama persis dengan ResNet50.

**Yang diuji** (peta occlusion tidak diulang):
1. Kinerja dalam generator, 5-fold, tiga kondisi.
2. Matriks kompresi: tiap model diuji pada tiga versi gambar uji yang sama.
3. Uji lintas generator, empat seed, tiga kondisi.
4. Uji tanpa warna: model simetris diuji pada gambar hitam-putih dan saturasi 50%.

**Aturan baca yang dikunci sebelum hasil:**
- **Aturan 1, perpindahan jalan pintas.** Terulang pada CLIP kalau, pada model satu sisi, gambar SD versi simetris dikenali minimal 10 poin lebih rendah daripada versi satu sisi **dan** karya manusia versi normal minimal 10 poin lebih rendah. Kalau kedua penurunan di bawah 5 poin, jalan pintas dinyatakan khas jaringan yang dilatih penuh. Di antaranya: dilaporkan apa adanya.
- **Aturan 2, lintas generator.** Pola sama dengan ResNet50 kalau, di kedua arah, rata-rata kondisi simetris lebih tinggi daripada satu sisi dan normal.
- **Aturan 3, warna.** Ketergantungan warna juga ada pada CLIP kalau gambar hitam-putih menurunkan pengenalan LD dan SD masing-masing minimal 20 poin.

Apa pun hasilnya, angka dilaporkan di naskah.

**Sebelum Run all:** menu **Runtime → Change runtime type → T4 GPU → Save**. Perkiraan 15 sampai 25 menit; sebagian besar untuk mengunduh model (sekitar 900 MB) dan mengambil fitur 15.000 gambar.

## 1. Persiapan dan cek GPU

In [ ]:
!pip -q install open_clip_torch

from google.colab import drive
drive.mount('/content/drive')

import os, io, json, time, random, shutil, zipfile, platform
import numpy as np, pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
from torchvision.transforms.functional import gaussian_blur
import cv2, scipy, sklearn, open_clip
from scipy import stats, ndimage
from PIL import Image
import PIL

# Pengaturan (sama dengan langkah 3 sampai 6; jangan diubah tanpa mencatatnya di log kerja)
SEED = 42
CLIP_ARCH, CLIP_WEIGHTS = 'ViT-L-14-quickgelu', 'openai'   # pembanding: CLIP yang dibekukan; varian quickgelu wajib untuk bobot OpenAI
PROBE_C = [0.001, 0.01, 0.1, 1.0]                 # kandidat regularisasi, dipilih lewat validasi silang di data latih
CROSS_SEEDS = [52, 1052, 2052, 3052]               # sama dengan uji lintas generator ResNet50
N_BOOT = 2000
INPUT = 224
PATCH, STRIDE = 32, 16
TOP = 0.20
N_RANDOM = 10
BLUR_KERNEL, BLUR_SIGMA = 21, 7.0
HUE_BINS, GLCM_LEVELS = 36, 32
CANNY_LOW, CANNY_HIGH = 100, 200
CHROMA_MIN = 0.10
MAX_EPOCHS, PATIENCE, BATCH, LR, LR_MIN, PRETRAINED = 30, 5, 32, 1e-4, 1e-6, True

PROJECT = Path('/content/drive/MyDrive/Paper2')
NOTES = PROJECT / 'catatan'
RESULTS = PROJECT / 'hasil'
RUNS = RESULTS / 'detektor_runs'
MODELS = RESULTS / 'models'
CLIPDIR = RESULTS / 'clip_pembanding_v2'   # folder baru: hasil versi pertama memakai aktivasi yang salah
CACHE = PROJECT / 'data_olahan'
LOCAL = Path('/content/work')
for p in [CLIPDIR, LOCAL]:
    p.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpu = torch.cuda.get_device_name(0) if device == 'cuda' else 'TIDAK ADA'
print('GPU:', gpu)
assert device == 'cuda', 'GPU belum aktif. Buka Runtime → Change runtime type → pilih T4 GPU → Save, lalu Run all lagi.'
USE_AMP = device == 'cuda'

env = (f'tanggal: {time.strftime("%Y-%m-%d %H:%M")}\npython: {platform.python_version()}\ntorch: {torch.__version__}\n'
       f'torchvision: {torchvision.__version__}\nnumpy: {np.__version__}\nscipy: {scipy.__version__}\nsklearn: {sklearn.__version__}\n'
       f'opencv: {cv2.__version__}\npillow: {PIL.__version__}\ngpu: {gpu}\n'
       f'open_clip: {open_clip.__version__}\nmodel: {CLIP_ARCH} ({CLIP_WEIGHTS}), dibekukan; pengklasifikasi: regresi logistik\n')
(NOTES / 'lingkungan_langkah11.txt').write_text(env)
print(env)

## 2. Data: tiga versi kompresi (sama dengan langkah 7)

In [ ]:
manifest = pd.read_csv(NOTES / 'manifest_ai_artbench_sample_v2.csv')
assert len(manifest) == 3000 and 'fold' in manifest.columns, 'Jalankan notebook langkah 2 dulu.'
manifest['y'] = (manifest['label'] == 'ai').astype(int)


local_raw = LOCAL / 'raw'
raw_zip = CACHE / 'ai_artbench_sample_raw.zip'
t0 = time.time()
if not local_raw.exists():
    shutil.copy(raw_zip, LOCAL / 'raw.zip')
    with zipfile.ZipFile(LOCAL / 'raw.zip') as z:
        z.extractall(local_raw)
manifest['raw_path'] = manifest.apply(
    lambda r: str(local_raw / r['source'] / r['style'] / Path(r['file_drive']).name), axis=1)

def enc(im, q, sub):
    buf = io.BytesIO(); im.save(buf, 'JPEG', quality=q, subsampling=sub, optimize=False); return buf.getvalue()
def dec(b):
    im = Image.open(io.BytesIO(b)); im.load(); return im.convert('RGB')
def square256(im):
    im = im.convert('RGB'); w, h = im.size
    if w != h:
        s = min(w, h); left, top = (w - s) // 2, (h - s) // 2
        im = im.crop((left, top, left + s, top + s))
    if im.size != (256, 256):
        im = im.resize((256, 256), Image.BICUBIC)
    return im

def make_version(row, version):
    with Image.open(row.raw_path) as im:
        im.load(); im = im.convert('RGB')
    if version == 'riwayat_disamakan' and row.source == 'human':
        im = dec(enc(im, 75, 2))
    im = square256(im)
    if version == 'simetris':
        im = dec(enc(im, 75, 2))
    b = enc(im, 90, 0)
    out = dec(b)
    if out.size != (INPUT, INPUT):
        out = out.resize((INPUT, INPUT), Image.BICUBIC)
    return np.asarray(out, dtype=np.uint8), len(b)

VERSIONS = ['normal', 'riwayat_disamakan', 'simetris']
XH, XT, SIZE = {}, {}, {}
for v in VERSIONS:
    with ThreadPoolExecutor(8) as ex:
        res = list(ex.map(lambda r: make_version(r, v), manifest.itertuples()))
    XH[v] = np.stack([a for a, _ in res]); SIZE[v] = np.array([s for _, s in res]) / 1024
    XT[v] = torch.from_numpy(XH[v]).permute(0, 3, 1, 2).contiguous()
idx_of = {s: i for i, s in enumerate(manifest['sample_id'])}
src_all, folds, y_all = manifest['source'].values, manifest['fold'].values, manifest['y'].values
strata_all = (manifest['source'] + '|' + manifest['style']).values
print(f'Data siap ({time.time() - t0:.0f} detik):', {v: XH[v].shape for v in VERSIONS})

## 3. Fitur CLIP untuk tiap versi gambar

In [ ]:
model = None
def get_model():
    global model
    if model is None:
        m, _, _ = open_clip.create_model_and_transforms(CLIP_ARCH, pretrained=CLIP_WEIGHTS)
        from open_clip.transformer import QuickGELU
        assert CLIP_WEIGHTS != 'openai' or any(isinstance(x, QuickGELU) for x in m.modules()), 'Aktivasi tidak cocok dengan bobot OpenAI. Kabari Claude.'
        model = m.to(device).eval()
    return model
MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=device).view(1, 3, 1, 1)
def to_gray(x):            # x: float (N, 3, H, W) dalam rentang 0-255; rumus sama persis dengan uji tanpa warna langkah 7
    g = 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]
    return g.expand(-1, 3, -1, -1)
def half_sat(x):
    return 0.5 * x + 0.5 * to_gray(x)
@torch.no_grad()
def embed_all(X, transform=None, batch=64):
    """Ubahan warna diterapkan per batch di GPU, supaya memori RAM tidak penuh."""
    out = []
    for i in range(0, len(X), batch):
        xb = X[i:i + batch].to(device).float()
        if transform is not None: xb = transform(xb)
        xb = (xb / 255.0).clamp(0, 1)
        if xb.shape[-1] != 224: xb = F.interpolate(xb, size=(224, 224), mode='bicubic', align_corners=False).clamp(0, 1)
        with torch.autocast('cuda', enabled=USE_AMP):
            f = get_model().encode_image((xb - MEAN) / STD)
        out.append(F.normalize(f.float(), dim=-1).cpu())
    return torch.cat(out).numpy()

JOBS = {v: (v, None) for v in VERSIONS}
JOBS['simetris_hitamputih'] = ('simetris', to_gray)
JOBS['simetris_saturasi50'] = ('simetris', half_sat)
EMB = {}
for name in JOBS:
    f = CLIPDIR / f'emb_{name}.npy'
    if f.exists():
        e = np.load(f)
        if e.shape[0] == len(manifest) and np.isfinite(e).all() and np.allclose(np.linalg.norm(e, axis=1), 1, atol=1e-3):
            EMB[name] = e; print(f'{name}: sudah ada, dimuat {e.shape}.'); continue
        print(f'{name}: file tersimpan tidak utuh, dihitung ulang.')
    t0 = time.time(); src, tf = JOBS[name]; EMB[name] = embed_all(XT[src], tf); np.save(f, EMB[name])
    print(f'{name}: {EMB[name].shape}, {time.time() - t0:.0f} detik')
assert all(EMB[k].shape == (len(manifest), EMB['normal'].shape[1]) for k in JOBS), 'Ukuran fitur tidak sama. Kabari Claude.'
print('Semua fitur siap:', {k: EMB[k].shape for k in JOBS})

## 4. Fungsi pengklasifikasi linear

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def fit_probe(X, y, seed):
    """Pilih C dengan validasi silang 3 lipatan di data latih (AUC), lalu latih ulang pada seluruh data latih."""
    skf = StratifiedKFold(3, shuffle=True, random_state=seed); scores = []
    for C in PROBE_C:
        s = []
        for tr, va in skf.split(X, y):
            m = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(X[tr], y[tr])
            s.append(roc_auc_score(y[va], m.predict_proba(X[va])[:, 1]))
        scores.append(np.mean(s))
    C = PROBE_C[int(np.argmax(scores))]
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(X, y), C

def rates(p, idx):
    y = y_all[idx]; s = src_all[idx]; pred = (p >= 0.5).astype(int)
    r = {'akurasi': float((pred == y).mean()), 'auc': float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else np.nan}
    for src, nm in (('human', 'manusia'), ('latent_diffusion', 'LD'), ('stable_diffusion', 'SD')):
        m = s == src
        if m.any(): r[f'benar_{nm}'] = float((pred[m] == y[m]).mean())
    return r

NAMA = {'normal': 'normal', 'riwayat_disamakan': 'satu sisi', 'simetris': 'simetris'}

## 5. Kinerja 5-fold, matriks kompresi, dan uji tanpa warna

In [ ]:
# E1 dan E2: 5-fold per kondisi, lalu tiap model diuji pada tiga versi gambar uji yang sama
rows, chosen = [], []
for cond in VERSIONS:
    for k in range(5):
        tr, te = np.where(folds != k)[0], np.where(folds == k)[0]
        m, C = fit_probe(EMB[cond][tr], y_all[tr], SEED + k); chosen.append({'kondisi': cond, 'fold': k, 'C': C})
        for v in VERSIONS:
            rows.append({'model': cond, 'gambar': v, 'fold': k, **rates(m.predict_proba(EMB[v][te])[:, 1], te)})
        if cond == 'simetris':
            for v in ('simetris_hitamputih', 'simetris_saturasi50'):
                rows.append({'model': cond, 'gambar': v, 'fold': k, **rates(m.predict_proba(EMB[v][te])[:, 1], te)})
MAT = pd.DataFrame(rows); MAT.to_csv(CLIPDIR / 'clip_matriks_dan_ablasi.csv', index=False)
pd.DataFrame(chosen).to_csv(CLIPDIR / 'clip_C_terpilih.csv', index=False)
print('Selesai:', len(MAT), 'baris. C terpilih:', pd.DataFrame(chosen).C.value_counts().to_dict())

## 6. Uji lintas generator, empat seed

In [ ]:
# E3: uji lintas generator, rancangan sama dengan ResNet50 (latih: 1.200 karya manusia + 750 gambar satu generator; uji: 300 karya manusia + 750 gambar generator lain)
SHORT = {'latent_diffusion': 'LD', 'stable_diffusion': 'SD'}
human_tr = np.where((src_all == 'human') & (folds != 4))[0]; human_te = np.where((src_all == 'human') & (folds == 4))[0]
rng = np.random.default_rng(SEED); rows = []
for cond in VERSIONS:
    for a, b in [('latent_diffusion', 'stable_diffusion'), ('stable_diffusion', 'latent_diffusion')]:
        tr = np.concatenate([human_tr, np.where(src_all == a)[0]]); te = np.concatenate([human_te, np.where(src_all == b)[0]])
        for seed in CROSS_SEEDS:
            m, C = fit_probe(EMB[cond][tr], y_all[tr], seed)
            p = m.predict_proba(EMB[cond][te])[:, 1]; ai = y_all[te] == 1
            hit = (p[ai] >= 0.5).astype(float)
            boot = hit[rng.integers(0, len(hit), (N_BOOT, len(hit)))].mean(1)
            rows.append({'kondisi': cond, 'arah': f'latih {SHORT[a]}, uji {SHORT[b]}', 'seed': seed, 'C': C,
                         'ai_dikenali': hit.mean(), 'ci_bawah': np.percentile(boot, 2.5), 'ci_atas': np.percentile(boot, 97.5),
                         **rates(p, te)})
CROSS = pd.DataFrame(rows); CROSS.to_csv(CLIPDIR / 'clip_lintas_generator.csv', index=False)
print('Selesai:', len(CROSS), 'baris.')

## 7. Ringkasan dan pemeriksaan aturan baca

In [ ]:
COLS = ['akurasi', 'auc', 'benar_manusia', 'benar_LD', 'benar_SD']
m = MAT.groupby(['model', 'gambar'])[COLS].mean()

print('C1. KINERJA DALAM GENERATOR (5-fold, rata-rata) — CLIP lawan ResNet50')
RES = {'normal': (0.988, 1.000), 'riwayat_disamakan': (0.980, 0.999), 'simetris': (0.976, 0.997)}
t = pd.DataFrame({NAMA[c]: {'CLIP akurasi': m.loc[(c, c), 'akurasi'], 'CLIP AUC': m.loc[(c, c), 'auc'],
                            'ResNet50 akurasi': RES[c][0], 'ResNet50 AUC': RES[c][1]} for c in VERSIONS}).T
print(t.round(3).to_string())

print('\nC2. MATRIKS KOMPRESI CLIP (benar per sumber; baris = kondisi latih, kolom = versi gambar uji)')
mm = m.loc[[(a, b) for a in VERSIONS for b in VERSIONS], ['benar_manusia', 'benar_LD', 'benar_SD', 'akurasi']]
mm.index = pd.MultiIndex.from_tuples([(NAMA[a], NAMA[b]) for a, b in mm.index], names=['model dilatih pada', 'gambar diuji'])
print(mm.round(3).to_string())

o = 'riwayat_disamakan'
drop_sd = 100 * (m.loc[(o, o), 'benar_SD'] - m.loc[(o, 'simetris'), 'benar_SD'])
drop_h = 100 * (m.loc[(o, o), 'benar_manusia'] - m.loc[(o, 'normal'), 'benar_manusia'])
print('\nC3. ATURAN BACA 1, perpindahan jalan pintas (model satu sisi; penurunan dalam poin persentase)')
print(f'   CLIP: SD {drop_sd:.1f} poin; manusia {drop_h:.1f} poin  |  ResNet50: 22,7 dan 16,7 (seed 1), 23,1 dan 18,3 (seed 2)')
if min(drop_sd, drop_h) >= 10: v1 = 'TERULANG pada CLIP (kedua penurunan minimal 10 poin)'
elif max(drop_sd, drop_h) < 5: v1 = 'TIDAK TERJADI pada CLIP (kedua penurunan di bawah 5 poin): jalan pintas khas jaringan yang dilatih penuh'
else: v1 = 'SEBAGIAN (di antara kedua batas): laporkan angka apa adanya'
print('   Kesimpulan:', v1)

print('\nC4. LINTAS GENERATOR (persen gambar AI dari generator tak dikenal yang dikenali; rata-rata 4 seed)')
ct = CROSS.groupby(['arah', 'kondisi']).agg(dikenali=('ai_dikenali', 'mean'), terendah=('ai_dikenali', 'min'), tertinggi=('ai_dikenali', 'max'),
                                            auc=('auc', 'mean'), manusia_benar=('benar_manusia', 'mean')).reset_index()
ct['kondisi'] = ct.kondisi.map(NAMA)
for c in ('dikenali', 'terendah', 'tertinggi', 'manusia_benar'): ct[c] = 100 * ct[c]
print(ct.round(1).to_string(index=False))
print('   ResNet50 (rata-rata 4 seed): latih LD uji SD: normal 1,2 / satu sisi 12,1 / simetris 21,4; latih SD uji LD: 2,0 / 2,1 / 10,6')
ok2 = True
for arah, g in ct.groupby('arah'):
    gi = g.set_index('kondisi')['dikenali']; ok = gi['simetris'] > gi['satu sisi'] and gi['simetris'] > gi['normal']; ok2 &= bool(ok)
    print(f"   {arah}: simetris {gi['simetris']:.1f}%, satu sisi {gi['satu sisi']:.1f}%, normal {gi['normal']:.1f}% -> {'simetris tertinggi' if ok else 'simetris BUKAN tertinggi'}")
print('   ATURAN BACA 2:', 'pola sama dengan ResNet50' if ok2 else 'pola BERBEDA dari ResNet50')
print('   Catatan: selang bootstrap 95% per seed ada di clip_lintas_generator.csv')

print('\nC5. ATURAN BACA 3, ketergantungan pada warna (model simetris; benar per sumber)')
ab = m.loc[[('simetris', v) for v in ('simetris', 'simetris_saturasi50', 'simetris_hitamputih')], ['akurasi', 'auc', 'benar_manusia', 'benar_LD', 'benar_SD']]
ab.index = ['tanpa ubahan', 'saturasi 50%', 'hitam-putih']; print(ab.round(3).to_string())
print('   ResNet50, hitam-putih: manusia 0,999; LD 0,317; SD 0,051')
d_ld = 100 * (ab.loc['tanpa ubahan', 'benar_LD'] - ab.loc['hitam-putih', 'benar_LD']); d_sd = 100 * (ab.loc['tanpa ubahan', 'benar_SD'] - ab.loc['hitam-putih', 'benar_SD'])
print(f"   Penurunan hitam-putih: LD {d_ld:.1f} poin, SD {d_sd:.1f} poin -> {'ketergantungan warna JUGA ada pada CLIP' if min(d_ld, d_sd) >= 20 else 'ketergantungan warna TIDAK sekuat ResNet50'} (syarat: keduanya minimal 20 poin)")
print('\nTersimpan di', CLIPDIR)

## Selesai

Kalau tabel C1 sampai C5 muncul, semua beres. Tekan **Cmd+S**, lalu kirim link Colab-nya ke Claude.